# धडा 18 (आगामी): पावती जी सिद्ध करतात की एक *माणूस* क्रिया अधिकृत केली

हा धडा सिद्ध करतो की **एजंट** काय केले आणि **गेट** ने काय ठरवलं. या नोटबुकमध्ये अजून एक हरवलेली अर्धी बाजू आहे: स्नानपात्र **नामांकित माणूस** ने **अचूक** क्रिया मान्य केल्याचा पुरावा — संपूर्ण प्रमाणिक क्रियेवर स्वतंत्र, माणुसकीने ठेवलेली सही, ऑफलाइन पडताळली गेली.

इथे दोन्ही वस्तू **धड्याच्या पावत्या तशाच काटेकोर स्वरूपाचा वापर करतात**: एक सपाट पेलोड ज्यात `type` फील्ड आहे, Ed25519 ने थेट प्रमाणिक JCS बाइट्सवर सही केली, ज्यासाठी एक संरचित `signature` ऑब्जेक्ट जोडलेला आहे (आणि सही केलेल्या बाइट्समधून वगळलेला). मान्यता पावती ही एक नवीन `type` (`human.approval.v1`) आहे ज्यामुळे क्रियेच्या प्रकाराबरोबर एकत्रित `verify_chain` द्वारे मुख्य नोटबुकमधील समान कोड मार्गाने दोन्ही वस्तू तपासता येतात. ही माणूस-मान्यता पावती शैक्षणिक स्वरूपात येथे परिभाषित केली गेली आहे, नव्हे तर draft-farley-acta-signed-receipts द्वारे परिभाषित केलेली पावती प्रकार नाही.

मुख्य नोटबुकमधील डेमो परीक्षकावर एक विचारपूर्वक सुधारणा: इथे परीक्षक `signature.key_id` ला एका **पिन केलेल्या की रजिस्ट्रीसह** पडताळतो, ना की पावतीत नेलेल्या सार्वजनिक कीवर अवलंबून राहतो. हाच तो उत्पादन दृष्टिकोन आहे ज्याची शिफारस हा धडा करत असलेल्या चेकलिस्टमध्ये आहे ("सार्वजनिक पडताळणी की प्रकाशित करा"), आणि त्यामुळे बनावट हे नाकारले जाते, तुम्ही तुमची स्वतःची की वापरून चुकवू शकत नाही.

हा नोटबुक शिकवतो की: **सही केलेली मान्यता म्हणजेच अधिकार नाही.** अधिकार फक्त तेव्हाच अस्तित्वात असतो जेव्हा मान्यता पावती आणि क्रिया पावती अजूनही एकाच प्रमाणिक क्रियेशी बांधलेल्या असतात, अंमलबजावणीच्या वेळी, धोरण आवृत्ती, की आणि कालबाह्यता जी अद्याप चालू असतात, आणि ज्याचं मान्यता अजून वापरलेली नाही. प्रत्येक अयशस्वी प्रयत्न एक **विशिष्ट कारण** नाकारतो, त्यामुळे तुम्हाला कळू शकते की *अधिकार कालबाह्य झाला आहे* की *अंमलात आणलेली क्रिया बदलली आहे*.


In [1]:
# These are already the Lesson 18 dependencies — no new packages.
# %pip install pynacl jcs
import base64, copy, hashlib
from jcs import canonicalize                      # RFC 8785 canonical JSON
from nacl.signing import SigningKey, VerifyKey
# CryptoError is the common base of BadSignatureError AND the ValueError pynacl
# raises for a wrong-length signature — catch the base so verification fails
# closed on ANY bad signature, not just the forged-but-correct-length one.
from nacl.exceptions import CryptoError

# Same helpers as the main notebook.
def b64url_nopad(data: bytes) -> str:
    return base64.urlsafe_b64encode(data).decode("ascii").rstrip("=")

def b64url_decode(s: str) -> bytes:
    return base64.urlsafe_b64decode(s + "=" * ((4 - len(s) % 4) % 4))

def sha256_canonical(obj) -> str:
    """SHA-256 of an object's JCS-canonical JSON form (same helper as the lesson)."""
    return f"sha256:{hashlib.sha256(canonicalize(obj)).hexdigest()}"

## अचूक क्रिया

मंजुरीची एकक आहे **कॅनॉनिकल क्रिया ऑब्जेक्ट** — "रिफंड मंजूर करा" सारख्या अस्पष्ट लेबल नाही, तर अचूक, पूर्णपणे निर्दिष्ट केलेली क्रिया. संपूर्ण ऑब्जेक्टवर स्वाक्षरी करणे (आणि त्यावरून एक डाइजेस्ट तयार करणे) हेच आम्हाला नंतर सिद्ध करण्यास अनुमती देते की मानवाने *हीच* क्रिया मंजूर केली आणि काहीही नव्हते.


In [2]:
action = {
    "action_type": "refund.issue",
    "params": {"order_id": "A-1029", "amount_usd": 4200, "to": "acct_88"},
    "policy_id": "refunds-v3",
}
print("action digest:", sha256_canonical(action))

action digest: sha256:fba342ad8447b491a089d7a09d4ac58f1a835c504e58f8d832db04f65bb62a25


## एक लिफाफा, दोन प्राधिकरणे

प्रत्येक पावती ही शिकवणीचा लिफाफा आहे: एक सपाट पेलोड `type` फील्डसह, तसेच `signature` ऑब्जेक्ट (`alg`, `sig`, `key_id`) जो स्वाक्षरी केलेल्या बाइट्सचा भाग **नाही**. `verify_envelope` हे दोन्ही पावती प्रकारांसाठी सामायिक संरचनीत आणि स्वाक्षरी तपासणी आहे; जी **पिन केलेली की नोंदणी** `signature.key_id` सोबत प्रमाणीकरण करते ती प्राधिकरणे वेगळी ठेवते:

- **मंजूरीची पावती** (`human.approval.v1`) — नाव असलेला मंजूरकर्ता, संपूर्ण कॅनॉनिकल क्रिया **आणि तिचा डाइजेस्ट**, `policy_version`, जारी आणि कालबाह्यता वेळा. एकदाच वापर चेन स्तरावर ट्रॅक केला जातो.
- **क्रिया पावती** (`agent.action.v1`) — एजंट ओळख, `run_id`, हीच कॅनॉनिकल क्रिया **डाइजेस्ट**, अंमलबजावणीचा परिणाम आणि वेळ, तसेच `parent_approval_ref`: मंजुरीची `receipt_hash`, जे शिकवणीच्या चेनमधील `previous_receipt_hash` सारखे आहे.

सामायिक `action_digest` फील्ड हे बाइंडिंगवर अवलंबून असलेल्या जोडणीचे ठिकाण आहे. `key_id` फक्त शोध सूचनेसाठी स्वाक्षरी ऑब्जेक्टमध्ये असते: त्याला वेगळ्या पिन केलेल्या कीकडे नेल्यास स्वाक्षरी तपासणी अयशस्वी होते, त्यामुळे त्याचा काही फायदा होत नाही.


In [3]:
# ---- pinned key registries: SEPARATE authorities, one envelope shape ----------
# Published out of band (the lesson checklist's JWK-Set pattern); the verifier
# NEVER trusts a key carried inside a receipt.
approver_sk = SigningKey.generate()
agent_sk    = SigningKey.generate()
APPROVER_KEYS = {"approver-key-1": b64url_nopad(bytes(approver_sk.verify_key))}
AGENT_KEYS    = {"agent-key-1":    b64url_nopad(bytes(agent_sk.verify_key))}

# The policy the approval is granted under. If this moves after approval, the
# approval is STALE even though its signature still verifies.
CURRENT_POLICY = {"policy_version": "refunds-v3"}

def sign_receipt(payload: dict, sk: SigningKey, key_id: str) -> dict:
    """Same signing pipeline as the lesson: Ed25519 over the canonical JCS
    bytes directly; the signature object is NOT part of the signed bytes."""
    canonical = canonicalize(payload)
    return {
        **payload,
        "signature": {"alg": "EdDSA", "sig": b64url_nopad(sk.sign(canonical).signature), "key_id": key_id},
    }

def verify_envelope(receipt, expected_type: str, trusted_keys: dict):
    """The SHARED verifier contract for any receipt kind; the caller picks which
    pinned registry (authority) resolves key_id. Fails closed on ANY
    attacker-shaped input: malformed is a refusal, never a crash."""
    if not isinstance(receipt, dict) or not isinstance(receipt.get("signature"), dict):
        return (False, "receipt malformed (not an object with a signature object)")
    sig_obj = receipt["signature"]
    if sig_obj.get("alg") != "EdDSA":
        return (False, "unsupported signature alg")
    if receipt.get("type") != expected_type:
        return (False, f"wrong receipt type (expected {expected_type})")
    # Key freshness is part of authority: a key_id rotated out of the pinned
    # registry confers nothing, even with a valid signature.
    pub = trusted_keys.get(sig_obj.get("key_id"))
    if pub is None:
        return (False, f"stale authority: key_id {sig_obj.get('key_id')!r} is not in the pinned registry (unknown or rotated out)")
    # Reconstruct the signed bytes exactly as the lesson does: everything except
    # the signature object, canonicalized and passed directly to Ed25519.
    payload = {k: v for k, v in receipt.items() if k != "signature"}
    try:
        canonical = canonicalize(payload)
        VerifyKey(b64url_decode(pub)).verify(canonical, b64url_decode(sig_obj.get("sig") or ""))
    except (CryptoError, TypeError, ValueError, base64.binascii.Error):
        return (False, "signature invalid (forged, tampered, or malformed)")
    return (True, "envelope ok")

def human_approval(action, approver_id, approved_at, sk=approver_sk,
                   key_id="approver-key-1", policy_version=None, expires_at=None):
    # deepcopy: the receipt must be an immutable record of what was approved —
    # a live reference would let a later mutation of `action` silently change the
    # signed payload. Digest the SNAPSHOT so the two can never diverge.
    approved_action = copy.deepcopy(action)
    payload = {
        "type": "human.approval.v1",
        "approver_id": approver_id,
        "action": approved_action,                       # the FULL canonical action
        "action_digest": sha256_canonical(approved_action),  # the join field
        "policy_version": policy_version or CURRENT_POLICY["policy_version"],
        "approved_at": approved_at,                      # ISO-8601 Zulu, like the lesson
        "expires_at": expires_at or approved_at[:11] + "23:59:59Z",
    }
    return sign_receipt(payload, sk, key_id)

In [4]:
approval = human_approval(action, "alice@ops (WebAuthn)", "2026-07-08T15:04:05Z",
                          expires_at="2026-07-08T15:19:05Z")
print(verify_envelope(approval, "human.approval.v1", APPROVER_KEYS))
print("binds digest:", approval["action_digest"][:23], "…  under", approval["policy_version"])

(True, 'envelope ok')
binds digest: sha256:fba342ad8447b491 …  under refunds-v3


## `verify_chain`: जिथे बाइंडिंग प्रत्यक्षात ठरवली जाते

`verify_chain` हा दोन स्वाक्षरी तपासण्यांवर आधारित सुलभ आवरण **नाही**. ही ती एकमेव जागा आहे जिथे सामायिक कॅनॉनिकल `action_digest`, मंजुरीची धोरण/की/कालबाह्यता **ताजीपणा**, आणि मंजुरीच्या **एकदाच वापर** यांची तपासणी एकत्र केली जाते, त्या क्रियेला *आता सादर करताना*.

प्रत्येक अपयश वेगळ्या **कारणाने** नाकारले जाते, त्यामुळे नाकारणीचा वाचक ठरवू शकतो की अधिकार जुना झाला आहे का (धोरण बदलले, की फिरवली, मंजुरीची कालबाह्यता, मंजुरी वापरली गेली) किंवा सादर केलेली क्रिया अजूनही वैध मंजुरीपासून बदलली आहे का (डाइजेस्ट बदल).


In [5]:
def receipt_hash(receipt: dict) -> str:
    """Content-derived id of a COMPLETE receipt (including its signature) —
    the same convention as previous_receipt_hash in the lesson's chain."""
    return sha256_canonical(receipt)

def agent_receipt(action, approval, executed_at, sk=agent_sk, key_id="agent-key-1"):
    executed_action = copy.deepcopy(action)    # snapshot, same reason as the approval
    payload = {
        "type": "agent.action.v1",
        "agent_id": "agent:refunds-bot",
        "run_id": "run-0001",
        "action": executed_action,
        "action_digest": sha256_canonical(executed_action),  # same join field
        "parent_approval_ref": receipt_hash(approval),
        "outcome": "performed",
        "executed_at": executed_at,
    }
    return sign_receipt(payload, sk, key_id)

_consumed = set()

def verify_chain(action_being_executed, approval, agent_rcpt, now: str):
    """One code path covers both receipt kinds (same envelope), then checks the
    things that only make sense TOGETHER: shared digest, freshness, consumption.
    `now` is an ISO-8601 Zulu timestamp; Zulu strings compare correctly as strings."""
    # 1. Shared envelope contract, separate authorities.
    ok, why = verify_envelope(approval, "human.approval.v1", APPROVER_KEYS)
    if not ok: return (False, f"approval: {why}")
    ok, why = verify_envelope(agent_rcpt, "agent.action.v1", AGENT_KEYS)
    if not ok: return (False, f"agent receipt: {why}")

    # 2. The join: BOTH receipts must bind the digest of the action being executed
    #    right now. A valid approval for a DIFFERENT action is substitution, and it
    #    gets its own reason — this is "the executed action changed".
    executing_digest = sha256_canonical(action_being_executed)
    if approval.get("action_digest") != executing_digest or approval.get("action") != action_being_executed:
        return (False, "digest substitution: the approval binds a different canonical action than the one being executed")
    if agent_rcpt.get("action_digest") != executing_digest or agent_rcpt.get("action") != action_being_executed:
        return (False, "digest substitution: the agent receipt binds a different canonical action than the one being executed")
    if agent_rcpt.get("parent_approval_ref") != receipt_hash(approval):
        return (False, "agent receipt is not bound to this approval")

    # 3. Freshness: a valid signature over stale authority is still a refusal —
    #    each staleness gets its own reason, distinct from substitution above.
    if approval.get("policy_version") != CURRENT_POLICY["policy_version"]:
        return (False, f"stale authority: approved under policy {approval.get('policy_version')!r}, current is {CURRENT_POLICY['policy_version']!r}")
    expires = approval.get("expires_at")
    if not isinstance(expires, str) or not expires or now >= expires:
        return (False, "stale authority: approval expired before execution")

    # 4. One-time consumption: an approval authorizes ONE execution.
    ref = receipt_hash(approval)
    if ref in _consumed:
        return (False, "approval already consumed (replay refused)")
    _consumed.add(ref)
    return (True, f"approved by {approval['approver_id']}, executed by {agent_rcpt['agent_id']}")

def execute(action, approval, agent_rcpt, now):
    ok, why = verify_chain(action, approval, agent_rcpt, now)
    return (ok, "executed" if ok else why)

receipt = agent_receipt(action, approval, "2026-07-08T15:04:06Z")
print(execute(action, approval, receipt, now="2026-07-08T15:04:07Z"))

(True, 'executed')


## बांधणी काय पकडते

खालील प्रत्येक प्रकरण **विशिष्ट कारणांसह** **बंद** होते. प्रथम ब्लॉक क्लासिक सेट आहे (छेडछाड, गोंधळलेला डिप्टी, पुनरावृत्ती, सत्ताधारीवर बनावट, अपूर्ण इनपुट). दुसरा ब्लॉक जो जोडपी मालमत्तेला खरे तर बनवतो प्रतिधारित केलेले नाही:

- **झालेल्या सत्ताधारी** — सही अद्याप वैध आहे, पण धोरण आवृत्ती बदलली, मंजुरी दणी की पिन केलेल्या नोंदणीतून वळविल्या गेले, किंवा अंमलबजावणीपूर्वी मंजुरी कालबाह्य झाली;
- **डाइजेस्ट बदल** — वैधपणे सही केलेल्या क्रियाकलाप रिसिट ज्याचा `parent_approval_ref` खऱ्या मंजुरीकडे निर्देश करतो, पण त्या मंजुरीचा सही क्रियाकलापाचा डाइजेस्ट प्रत्यक्षात चालविल्या जाणाऱ्या क्रियाकलापाशी जुळत नाही.


In [6]:
NOW = "2026-07-08T15:05:00Z"

# 1. tamper: change the amount after approval — the executed action changed.
tampered = {**action, "params": {**action["params"], "amount_usd": 9900}}
print("tamper              ->", verify_chain(tampered, approval, agent_receipt(tampered, approval, NOW), NOW))

# 2. confused deputy: valid approval for action A, presented to execute action B.
action_b = {**action, "action_type": "wire.send"}
print("confused-deputy     ->", verify_chain(action_b, approval, agent_receipt(action_b, approval, NOW), NOW))

# 3. replay: the approval was consumed by the successful execution above.
print("replay              ->", execute(action, approval, agent_receipt(action, approval, NOW), NOW))

# 4. forged approval: attacker signs with their own key but claims a pinned key_id.
mallory_sk = SigningKey.generate()
forged = human_approval(action, "mallory", NOW, sk=mallory_sk)
print("forged-approval     ->", verify_chain(action, forged, agent_receipt(action, forged, NOW), NOW))

# A fresh, un-consumed approval so the agent-side cases fail on their OWN check.
fresh = human_approval(action, "alice@ops (WebAuthn)", NOW, expires_at="2026-07-08T15:20:00Z")

# 5. self-minted agent receipt: attacker's own agent key, refused by the pinned registry.
mallory_agent = agent_receipt(action, fresh, NOW, sk=SigningKey.generate())
print("self-minted-agent   ->", verify_chain(action, fresh, mallory_agent, NOW))

# 6. wrong-action agent receipt: real agent key, but the receipt binds a different action.
wrong_action = {**action, "params": {**action["params"], "amount_usd": 9900}}
print("wrong-action-agent  ->", verify_chain(action, fresh, agent_receipt(wrong_action, fresh, NOW), NOW))

# 7. malformed input: structurally broken receipts refuse cleanly, they never crash.
print("malformed-approval  ->", verify_chain(action, {"type": "human.approval.v1"}, agent_receipt(action, fresh, NOW), NOW))
print("malformed-agent     ->", verify_chain(action, fresh, {"nope": "not a receipt"}, NOW))

# 8. wrong-length signature: valid base64, not 64 bytes — refused, not crashed.
badlen = {**fresh, "signature": {**fresh["signature"], "sig": "AAAA"}}
print("wrong-len-sig       ->", verify_chain(action, badlen, agent_receipt(action, fresh, NOW), NOW))

# 9. non-object receipt: a list refuses cleanly instead of raising AttributeError.
print("nonobject-receipt   ->", verify_chain(action, [1, 2], agent_receipt(action, fresh, NOW), NOW))

print()
print("--- the two negative controls that make the property real ---")

# 10. STALE POLICY: signature still valid, but policy moved between approval and
#     execution. Authority is decided at execution time, not signing time.
CURRENT_POLICY["policy_version"] = "refunds-v4"
print("stale-policy        ->", verify_chain(action, fresh, agent_receipt(action, fresh, NOW), NOW))
CURRENT_POLICY["policy_version"] = "refunds-v3"   # restore for the cases below

# 11. STALE KEY: the approver key is rotated out of the pinned registry after
#     signing. The signature bytes still verify against the old key — but the old
#     key no longer confers authority.
rotated_out = APPROVER_KEYS.pop("approver-key-1")
print("stale-key           ->", verify_chain(action, fresh, agent_receipt(action, fresh, NOW), NOW))
APPROVER_KEYS["approver-key-1"] = rotated_out     # restore

# 12. EXPIRED: approval was valid when signed, but execution came too late.
expired = human_approval(action, "alice@ops (WebAuthn)", "2026-07-08T14:00:00Z",
                         expires_at="2026-07-08T14:01:00Z")
print("expired-approval    ->", verify_chain(action, expired, agent_receipt(action, expired, NOW), NOW))

# 13. DIGEST SUBSTITUTION: a validly signed agent receipt whose parent_approval_ref
#     points at a REAL approval — but that approval binds action B, and the agent
#     is executing action A. Distinct reason from every staleness above.
approval_b = human_approval(action_b, "alice@ops (WebAuthn)", NOW, expires_at="2026-07-08T15:20:00Z")
substituted = agent_receipt(action, approval_b, NOW)   # executing `action`, ref -> approval of action_b
print("digest-substitution ->", verify_chain(action, approval_b, substituted, NOW))

tamper              -> (False, 'digest substitution: the approval binds a different canonical action than the one being executed')
confused-deputy     -> (False, 'digest substitution: the approval binds a different canonical action than the one being executed')
replay              -> (False, 'approval already consumed (replay refused)')
forged-approval     -> (False, 'approval: signature invalid (forged, tampered, or malformed)')
self-minted-agent   -> (False, 'agent receipt: signature invalid (forged, tampered, or malformed)')
wrong-action-agent  -> (False, 'digest substitution: the agent receipt binds a different canonical action than the one being executed')
malformed-approval  -> (False, 'approval: receipt malformed (not an object with a signature object)')
malformed-agent     -> (False, 'agent receipt: receipt malformed (not an object with a signature object)')
wrong-len-sig       -> (False, 'approval: signature invalid (forged, tampered, or malformed)')
nonobject-receipt   -> (Fa

## हे काय सिद्ध करते — आणि काय नाही  

**सिद्ध करते:** एका नावाजलेल्या व्यक्तींनी मान्यता दिली *या अचूक मानक क्रियेवर* (पूर्ण क्रिया + डाइजेस्ट, पिन केलेल्या रजिस्ट्रीमधून सोडवलेल्या कीने स्वाक्षरी केलेली), आणि एजंटने *अचूक ती मान्य केलेली क्रिया* पार पाडली (ताच डाइजेस्ट, `receipt_hash` ने मान्यतेशी बांधलेले पावती, लेसनच्या स्वतःच्या चेन संहितेप्रमाणे) — जेव्हाही मान्यतेच्या धोरण आवृत्ती, की आणि कालबाह्यता अद्ययावत होते, फक्त एकदा. जर कोणतीही बाजू बदलली, तर चेन बंद पडते, आणि नकाराचा कारण तुम्हाला सांगतो **कोणती** मालमत्ता खराब झाली: जुनी अधिकृतता किंवा बदललेली क्रिया.  

**सिद्ध होत नाही:** की मान्यता UI माणसाला जे देत होती तेच त्यांनी स्वाक्षरी केली आहे (WYSIWYS हे स्वतःचं एक समस्या आहे), की की वळणापूर्वी जबरदस्तीने किंवा चोरीने घेतलेली नाही, किंवा की खालील परिणाम क्रियेशी जुळले आहेत. स्वाक्षरी केलेले ≠ अधिकृत: जुना धोरण, वळवलेली की, कालबाह्य वेळा, किंवा भिन्न डाइजेस्ट या सर्वांच्या वैध स्वाक्षरीने येथे काहीही दिलं जात नाही.  

दोन पावती प्रकार लेसनच्या कागदपत्राचा आणि एका `verify_chain` कोड मार्गाचा उद्देशाने वापर करतात: मुख्य नोटबुकमधील क्रिया पावतासाठी बांधलेली लिंक मान्यतेच्या तपासणीसाठी वापरली जाते. एकच तपासणी करणारं करारपत्र, स्वतंत्र पिन केलेल्या अधिकरणांसह, मानक क्रिया डाइजेस्टने जोडलेले आणि त्यापलीकडे काही नाही.  


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**अस्वीकरण**:
हा दस्तऐवज AI भाषांतर सेवा [Co-op Translator](https://github.com/Azure/co-op-translator) चा वापर करून अनुवादित केला आहे. जरी आम्ही अचूकतेसाठी प्रयत्न करतो, तरी कृपया लक्षात घ्या की स्वयंचलित भाषांतरांमध्ये त्रुटी किंवा अचूकतेची कमतरता असू शकते. मूळ दस्तऐवज त्याच्या मूळ भाषेत अधिकृत स्रोत मानला पाहिजे. महत्त्वाची माहिती असल्यास, व्यावसायिक मानवी भाषांतराची शिफारस केली जाते. या भाषांतराच्या वापरामुळे उद्भवणाऱ्या कोणत्याही गैरसमज किंवा चुकीच्या अर्थलावणीसाठी आम्ही जबाबदार नाही.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
